# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [12]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [13]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
len(parquet_files)

3337

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [15]:
import pandas as pd

dd_px = dd.read_parquet(parquet_files).set_index("ticker")

def add_lags(ticker_df):
    # Put the ticker's rows in date order so shift(1) picks up the previous trading day.
    ticker_df = ticker_df.sort_values("Date")
    return ticker_df.assign(
        Close_lag_1=ticker_df["Close"].shift(1),
        Adj_Close_lag_1=ticker_df["Adj Close"].shift(1),
    )

meta = dd_px._meta.assign(
    Close_lag_1=pd.Series(dtype="float64"),
    Adj_Close_lag_1=pd.Series(dtype="float64"),
)

dd_feat = (
    dd_px
    .groupby("ticker", group_keys=False)
    .apply(add_lags, meta=meta)
    .assign(
        returns=lambda x: x["Close"] / x["Close_lag_1"] - 1,
        hi_lo_range=lambda x: x["High"] - x["Low"],
    )
)
dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=90,,,,,,,,,,,,,
ACN,datetime64[us],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [16]:
df_feat = dd_feat.compute().sort_values(["ticker", "Date"])

df_feat["returns_ma_10"] = (
    df_feat.groupby("ticker")["returns"]
    .transform(lambda r: r.rolling(10).mean())
)
df_feat.head(15)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
ticker,,,,,,,,,,,,,,
ACN,2002-01-02,26.900000,26.910000,25.950001,26.209999,19.703959,900500.0,ACN.csv,2002,NaN,NaN,NaN,0.959999,NaN
ACN,2002-01-03,26.230000,26.270000,25.299999,25.389999,19.087507,698200.0,ACN.csv,2002,26.209999,19.703959,-0.031286,0.970001,NaN
ACN,2002-01-04,25.389999,28.200001,25.240000,27.700001,20.824106,2708500.0,ACN.csv,2002,25.389999,19.087507,0.090981,2.960001,NaN
ACN,2002-01-07,27.770000,27.799999,26.450001,26.450001,19.884382,1113600.0,ACN.csv,2002,27.700001,20.824106,-0.045126,1.349998,NaN
ACN,2002-01-08,26.459999,27.299999,24.120001,27.280001,20.508362,1746900.0,ACN.csv,2002,26.450001,19.884382,0.031380,3.179998,NaN
ACN,2002-01-09,27.000000,28.340000,26.260000,27.570000,20.726372,3008600.0,ACN.csv,2002,27.280001,20.508362,0.010630,2.080000,NaN
ACN,2002-01-10,27.980000,28.150000,27.260000,27.820000,20.914314,2384900.0,ACN.csv,2002,27.570000,20.726372,0.009068,0.889999,NaN
ACN,2002-01-11,27.600000,27.750000,26.010000,26.469999,19.899424,3910300.0,ACN.csv,2002,27.820000,20.914314,-0.048526,1.740000,NaN
ACN,2002-01-14,25.600000,25.650000,24.809999,24.940001,18.749210,4285500.0,ACN.csv,2002,26.469999,19.899424,-0.057801,0.840000,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

It was not necessary. Dask can calculate rolling statistics itself, one partition at a time, without any extra shuffling of the data.

+ Would it have been better to do it in Dask? Why?

For this assignment, pandas was a reasonable choice, as the pandas code for a grouped rolling window is simpler, and Dask's task scheduling adds coverage that doesn't pay off on small data.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.